# 09.05 · Custom workflows and choosing an architecture

**Module 09 · Multi-Agent Systems** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lessons 09.01–09.04; Module 07 (graphs); Module 08 (interrupts, subgraphs). |
| 🔑 Keys | none |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Build** a custom workflow as a LangGraph graph that mixes deterministic steps with model/agent steps (Pattern C: the refund path).
2. **Decide** which steps must be code (policy, validation, approval) and which may be a model.
3. **Embed** the workflow next to agents: refunds through the graph, everything else through the handoffs desk.
4. **Compare** every design of this module on the 15-conversation test set in one table.
5. **Write** an architecture decision record (ADR) from measured numbers, including when a harness (Module 11) is the better tool.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Custom workflow](https://docs.langchain.com/oss/python/langchain/multi-agent/custom-workflow) | An agent called inside any LangGraph node; the rewrite → retrieve → agent example |
| [Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents) | Prompt chaining, routing, parallelization, orchestrator-worker, evaluator-optimizer |
| [Runtimes, frameworks, and harnesses](https://docs.langchain.com/oss/python/concepts/products) | When a harness (Deep Agents) beats a hand-built graph |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/09-multi-agent-systems/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Offline: the shared desk (Lesson 09.01 Part 2).
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from mastery.agents import UsageMeter
from mastery.agents.support import (SupportDesk, evaluate, faq_search, load_conversations, make_support_tools,
                                    offline_model, run_conversation)

desk = SupportDesk.load()
search = faq_search()
tools = make_support_tools(desk, search)
conversations = load_conversations()
by_id = {c["id"]: c for c in conversations}

> This lesson is fully 🧪 offline: the only model in the custom workflow writes the reply, and offline it returns the facts unchanged. With `GROQ_API_KEY` you can pass `get_model("fast")` as `writer` and nothing else changes.

---
## Part 1 · A workflow you write yourself (the docs' shape)

The docs' core idea: **any node can call a model or a whole agent**. Smallest useful version: a deterministic **retrieve** node feeding an **answer** node. No tool call to decide whether to search: the graph always searches.

In [ ]:
# 🧪 Offline: retrieve (code) → answer (model), as a two-node StateGraph.
from typing import TypedDict

from langgraph.graph import END, START, StateGraph
from mastery.testing import ScriptedChatModel


class FaqState(TypedDict):
    question: str
    passages: list[str]
    answer: str


def retrieve(state: FaqState) -> dict:                       # deterministic: no model
    return {"passages": [d.page_content for d in search(state["question"])]}


writer = ScriptedChatModel(responses=[lambda msgs: msgs[-1].text.split("PASSAGES:", 1)[1].strip()[:160]],
                           repeat_last=True)                  # offline: quote the first passage


def answer(state: FaqState) -> dict:                          # model step
    prompt = f"Answer from the passages only.\nQUESTION: {state['question']}\nPASSAGES: " + "\n".join(state["passages"])
    return {"answer": writer.invoke(prompt).text}


faq_flow = (StateGraph(FaqState).add_node("retrieve", retrieve).add_node("answer", answer)
            .add_edge(START, "retrieve").add_edge("retrieve", "answer").add_edge("answer", END).compile())
out = faq_flow.invoke({"question": "How long is a password reset link valid?"})
print(len(out["passages"]), "passages |", out["answer"])

**You should now see** `2 passages | # Orbita Cloud account FAQ` and, on the next lines, `## Resetting your password` and `... The reset link is valid for 60 minutes. ...` (the offline writer quotes the top passage): one model call, always one search.

Compare with the technical specialist (09.02): it **decides** whether to search (2 calls). A workflow trades that flexibility for predictability: fixed cost, fixed order, nothing skipped.

---
## Part 2 · Pattern C: the refund path as a custom graph

Refunds have rules that must **always** hold: the invoice belongs to the customer, the amount is within the invoice, no double refunds, a manager approves above $50. In the agent patterns these rules live in a prompt and a tool. In Pattern C they are **graph steps**:

| Node | Kind | Does |
|---|---|---|
| `parse` | code | Customer ID, invoice and amount from the conversation (regex) |
| `validate` | code | Ownership, amount range, not already refunded → else straight to `reply` |
| `approve` | code + `interrupt()` | Above $50: pause for the manager (approve / edit amount / reject) |
| `issue` | code | Record the refund |
| `reply` | **model** | Turn the facts into a friendly message (numbers kept exactly) |

`mastery.agents.support.build_refund_workflow(desk, writer)` is that graph.

In [ ]:
# 🧪 Offline: the refund workflow's structure.
from mastery.agents.support import _echo_writer, build_refund_workflow

refund_flow = build_refund_workflow(desk, _echo_writer(), checkpointer=InMemorySaver())
print(refund_flow.get_graph().draw_mermaid())

**You should now see** Mermaid text with `parse --> validate`, dotted (conditional) edges `validate -.-> approve` and `validate -.-> reply`, `approve -.-> issue` / `approve -.-> reply`, and `issue --> reply`. Paste it into [mermaid.live](https://mermaid.live) to see the picture (Lesson 07.01 drew graphs as PNGs the same way).

In [ ]:
# 🧪 Offline: three refunds through the workflow: small, big (manager edits the amount), invalid.
cases = [("small", "I'm C-1003, refund INV-3001 ($19)", None),
         ("big", "C-1002: refund $199 for INV-2001", {"action": "edit", "text": "150"}),
         ("invalid", "C-1001: refund $49 for INV-2001", None)]
desk.reset()
for label, text, decision in cases:
    config = {"configurable": {"thread_id": f"refund-{label}"}}
    meter = UsageMeter()
    out = refund_flow.invoke({"messages": [{"role": "user", "content": text}]}, {**config, "callbacks": [meter]})
    if "__interrupt__" in out:
        print(f"{label:<8} ⏸️  {out['__interrupt__'][0].value['question']}")
        out = refund_flow.invoke(Command(resume=decision), {**config, "callbacks": [meter]})
    print(f"{label:<8} calls={meter.model_calls} | {out['messages'][-1].content}")
print("refunds:", [(r["invoice_id"], r["amount"], r["approved_by"]) for r in desk.refunds])

**You should now see**

- `small calls=1 | Refunded $19.00 for INV-3001 (approved by policy) ...`;
- `big ⏸️ Approve a $199.00 refund (over $50)? ...`, then `big calls=1 | Refunded $150.00 for INV-2001 (approved by billing manager) ...`;
- `invalid calls=1 | I could not refund this request: invoice INV-2001 does not belong to C-1001.`;
- `refunds: [('INV-3001', 19.0, 'policy'), ('INV-2001', 150.0, 'billing manager')]`.

### What just happened?

**One** model call per refund (the reply), against 2–4 in the agent patterns, and the policy can't be talked around: there is no prompt to jailbreak between `validate` and `issue`. One input still comes from the chat, though: **whose** invoice it is. That's why `parse` takes the customer from the login (`runtime.context`, Lesson 09.02 Part 8) when there is one, and refuses a refund for another customer's invoice. The price: the workflow only handles what you coded. "Refund last month's invoice" (no invoice ID) fails at `validate` with `missing invoice_id` instead of the agent asking a clarifying question. That's why Pattern C is a **path**, not the whole desk.

---
## Part 3 · Embed it: refunds through the graph, the rest through agents

`build_custom_desk` puts a tiny router in front: a message that is only billing **and** asks for a refund goes to the refund workflow (as a **subgraph node**, sharing `messages`); everything else goes to the handoffs desk (called **inside a node**, the docs' "agent in a node"). One checkpointer, at the top.

In [ ]:
# 🧪 Offline: which path each test conversation takes, and a paused refund inside the embedded graph.
from mastery.agents.support import build_custom_desk, is_refund_request

paths = {c["id"]: ["refunds" if is_refund_request(t) else "agents" for t in c["turns"]] for c in conversations}
print({k: v for k, v in paths.items() if "refunds" in v})

custom = build_custom_desk(offline_model, tools, desk, checkpointer=InMemorySaver())
desk.reset()
config = {"configurable": {"thread_id": "custom-S02"}}
out = custom.invoke({"messages": [{"role": "user", "content": by_id["S02"]["turns"][0]}]}, config)
print("paused in:", custom.get_state(config, subgraphs=True).tasks[0].state.next)
out = custom.invoke(Command(resume="approve"), config)
print(out["messages"][-1].content)

**You should now see** six conversations with a `refunds` turn (S01, S02, S03, S04, S12 turn 2, S15 turn 2; **not** S14, which also has a technical question), then `paused in: ('approve',)` and `Refunded $199.00 for INV-2001 (approved by billing manager) ...`.

Because the workflow is a **subgraph node**, `get_state(..., subgraphs=True)` shows exactly where it paused (`approve`), which you could not see for subagents inside tools (09.02).

---
## Part 4 · Every design, one table

In [ ]:
# 🧪 Offline: all designs from this module on the 15 conversations; saved for your ADR.
import json

from mastery.agents.support import (build_handoffs_desk, build_router_desk, build_single_agent_baseline,
                                    build_single_agent_desk, build_skills_desk, build_subagents_desk)

DESIGNS = {
    "single agent": lambda: build_single_agent_baseline(offline_model(), tools, checkpointer=InMemorySaver()),
    "handoffs (1 agent)": lambda: build_single_agent_desk(offline_model(), tools, checkpointer=InMemorySaver()),
    "handoffs (graph)": lambda: build_handoffs_desk(offline_model, tools, checkpointer=InMemorySaver()),
    "subagents": lambda: build_subagents_desk(offline_model, tools, checkpointer=InMemorySaver()),
    "skills": lambda: build_skills_desk(offline_model(), tools, checkpointer=InMemorySaver()),
    "router": lambda: build_router_desk(offline_model, tools, checkpointer=InMemorySaver()),
    "custom (C + handoffs)": lambda: build_custom_desk(offline_model, tools, desk, checkpointer=InMemorySaver()),
}
results = []
for name, build in DESIGNS.items():
    meter = UsageMeter()
    report = evaluate(build(), conversations, desk, meter=meter)
    refund_convs = sum("issue_refund" in r["outcome"]["tools"] for r in report["rows"])
    results.append({"design": name, "passed": f"{report['passed']}/15", "model calls": meter.model_calls,
                    "tokens": meter.total_tokens, "refund convs": refund_convs,
                    "policy ok": all(r["refund_policy"] for r in report["rows"])})
table([{**r, "tokens": f"{r['tokens']:,}"} for r in results],
      ["design", "passed", "model calls", "tokens", "refund convs", "policy ok"])

out_dir = REPO_ROOT / "data" / "outputs" / "m09"
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "designs-offline.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
print("saved", (out_dir / "designs-offline.json").relative_to(REPO_ROOT))

**You should now see** seven rows, all `15/15` with `policy ok` `True` and 7 refund conversations each; model calls **36 / 53 / 53 / 71 / 53 / 36 / 42** (single agent / handoffs ×2 / subagents / skills / router / custom); tokens from about 13K (router) to 57K (skills); and `saved data/outputs/m09/designs-offline.json`.

### What just happened?

Offline, every design is **correct**, so the decision is about cost and control. The router and the single agent are cheapest; the custom design is cheaper than plain handoffs (42 vs 53) because refunds take one model call instead of three. Subagents are the most expensive but the only design where one agent reviews every answer. With real models, pass rates **will** differ: that's what Project 09 measures for handoffs vs subagents.

---
## Part 5 · Write the ADR

An **ADR** (architecture decision record) is one page: the context, the options, the evidence, the decision and its consequences. Generate the skeleton with your numbers, then write the reasoning yourself. Log the final decision in [`docs/decision-log.md`](../../docs/decision-log.md) too.

In [ ]:
# 🧪 Offline: an ADR skeleton with the measured table filled in.
rows_md = "\n".join(f"| {r['design']} | {r['passed']} | {r['model calls']} | {r['tokens']:,} | {'yes' if r['policy ok'] else 'NO'} |"
                     for r in results)
adr = f"""# ADR-09: Architecture for the Orbita Cloud support desk

**Status:** proposed · **Date:** (today) · **Deciders:** (you)

## Context
Three domains (billing, technical, account), 7 tools, a refund policy that needs a manager above $50,
15 test conversations (one-shot, multi-turn, domain switch, multi-domain).

## Options and evidence (offline, 15 conversations)
| Design | Passed | Model calls | Tokens | Refund policy held |
|---|---|---|---|---|
{rows_md}

LIVE (gpt-oss-20b): (fill in from Project 09)

## Decision
(Which design, and why: name the requirement each choice serves.)

## Consequences
- Cost per conversation: ...
- What gets harder: ...
- When to revisit (e.g. > N tools, a new domain, a pass rate below X): ...

## Harness?
(Which specialist would gain from a full harness such as Deep Agents: planning, a filesystem, its own subagents? Module 11.)
"""
(out_dir / "ADR-09-draft.md").write_text(adr, encoding="utf-8")
print("\n".join(adr.splitlines()[:16]))

**You should now see** the first lines of the ADR: title, status, context, and the evidence table with your seven rows; the full skeleton is in `data/outputs/m09/ADR-09-draft.md` (git-ignored; copy it to your notes or `docs/` when it's written).

**A reasonable recommendation for this desk** (yours may differ, if you can defend it):

- **Default:** handoffs in **one agent** (middleware steps): cheap repeat turns, one state, simple tracing.
- **Refunds:** Pattern C, because money rules belong in code and the workflow is the cheapest path.
- **Not now:** subagents (pay 2 coordinator calls per turn for review you don't need yet); skills (one context for all domains).
- **Revisit** when a domain gets more than ~10 tools or another team owns it (→ subagents), or when classification is reliable enough to route without an agent (→ router).

---
## Part 6 · Preview: when a harness beats your graph

Everything in this module is **hand-assembled** from `create_agent` and LangGraph. A **harness** (Deep Agents, Module 11) ships the parts you'd otherwise build for long, open-ended tasks: a planning tool (todo list), a virtual filesystem for notes and large tool results, built-in subagents with isolated context, context summarization, and human approval on tools.

| Need | Hand-built (this module) | Harness (Module 11) |
|---|---|---|
| Fixed policy, few steps (refunds) | ✅ custom workflow | overkill |
| Short Q&A per domain | ✅ handoffs / router | overkill |
| Open-ended investigation ("why does station X lose data every night?": logs, firmware notes, many searches, a written report) | lots of plumbing | ✅ planning + files + subagents |

On this desk the **technical** specialist is the candidate: its hard cases are multi-step investigations, not lookups. Project 09 asks you to note that; Module 11 rebuilds it as a deep agent.

---
## ⚠️ Pitfalls and troubleshooting (Windows Git Bash included)

| Symptom | Cause | Fix |
|---|---|---|
| `InvalidUpdateError` / missing key when the workflow runs as a subgraph | Parent and subgraph share no key, or the subgraph expects keys the parent never sets | Share `messages`; initialize the subgraph's own keys in its first node (`parse` does) |
| The workflow refunds a different invoice than the customer meant | Regex parsing picked the wrong ID | Validate against the conversation and the customer record; ask when ambiguous; add the case to the test set |
| Resume after approval refunds twice | A side effect **before** `interrupt()` in the same node | Side effects after the decision (`issue` is its own node) |
| A manager's "edit" refunds $1,990 on a $199 invoice, or −$5 | The human's answer skipped the checks the agent's request went through | Validate a human's edit like any other input: same `0 < amount ≤ invoice` rule, re-ask with an error on a bad answer (`manager_review` does), and log the amount actually refunded |
| The model "rewrites" amounts in the reply | Free-form generation | Keep facts in code, tell the writer to keep numbers, check the reply contains them (the eval does) |
| `get_state(subgraphs=True)` shows no tasks | Not paused, or the graph was called inside a **tool** (subagents): LangGraph only finds graphs added as nodes or invoked from node functions | Add compiled graphs as nodes (or call them from a node function) when you want to inspect their state |

## ⚠️ Production notes

> ⚠️ **Production note: put money, permissions and compliance in code.**
> A rule that must always hold (limits, approvals, ownership) belongs in a graph step or a tool's code, never only in a prompt. Agents can propose; code decides.

> ⚠️ **Production note: an ADR is a living document.**
> Re-run the comparison when models, prompts or the test set change, and update the ADR's evidence table. A decision without the numbers that justified it can't be revisited safely.

## 🏋️ Exercises

1. **(Easy)** Run the refund workflow on `"C-1001: refund INV-1003"` (no amount). Which node stops it, and what does the customer read?
2. **(Medium)** Add a `duplicate_check` node after `validate`: if the invoice's note says `duplicate charge`, skip the manager even above $50 (the FAQ says duplicates are always refunded). Test with an over-threshold duplicate you add to `customers.json`'s copy in memory.
3. **(Medium)** Change `build_custom_desk`'s `pick` to also send **account** requests to a small deterministic workflow (unlock → reply). How many model calls does the test set need now?
4. **(Stretch)** Fill in the ADR with live numbers: run Part 4's loop with `get_model("fast")` on 8 conversations that cover every route (S02, S03, S06, S10, S12, S13, S14, S15) for 3 designs (💸 roughly 70–110 calls).

<details>
<summary><b>Show solutions</b></summary>

**1.** `validate` sets `problem = "missing amount"`, the graph goes to `reply`: `I could not refund this request: missing amount.` An agent would have asked "how much?"; add that question to `reply`'s facts when `missing` is the problem.

**2.**
```python
def duplicate_check(state):
    r = state["request"]
    inv = next(i for i in desk.customers[r["customer_id"]]["invoices"] if i["id"] == r["invoice_id"])
    return {"approved_by": "duplicate policy"} if inv.get("note") == "duplicate charge" else {}
# route: validate → duplicate_check → (issue if approved_by else approve)
desk.customers["C-1001"]["invoices"].append({"id": "INV-1009", "amount": 99.0, "date": "2026-10-01",
                                             "note": "duplicate charge"})
```
Build your own copy of the graph (copy `build_refund_workflow` from `src/mastery/agents/support.py` into a cell and add the node).

**3.** Unlock and password reset become 1 call each (the reply) instead of 3 (triage + account ×2): 4 fewer calls (S09 and S10 each go from 3 to 1): **38** in total, against 42 now. The email change needs the new address parsed and confirmed; keep it on the agent path.

**4.** Reuse `DESIGNS` with `live_model = lambda role: get_model("fast", temperature=0, reasoning_effort="low")` in place of `offline_model` (and `live_model("x")` where a single model is passed), and run it on `[c for c in conversations if c["id"] in {"S02", "S03", "S06", "S10", "S12", "S13", "S14", "S15"}]` (not `limit=5`: the first five conversations are all billing, so they hide every routing failure). Paste the table under **LIVE** in the ADR.
</details>

## 🔎 Docs drill (5 min)

When do the docs recommend a custom workflow over the built-in patterns, and which LangGraph features does that page say you combine?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

Open [Custom workflow](https://docs.langchain.com/oss/python/langchain/multi-agent/custom-workflow): a custom workflow fits when none of the patterns matches your control flow, and you mix deterministic steps with agentic ones in one `StateGraph` (agents as nodes, plain functions as nodes, conditional edges). Compare with Part 1: the refund path is exactly that mix.

</details>

## ✅ Checkpoint

- [ ] I built a workflow with deterministic nodes and a model node, and explained what it trades away.
- [ ] Refunds go through a graph whose policy is code, with an `interrupt()` above $50.
- [ ] I embedded the workflow next to an agent desk with one checkpointer and inspected the paused subgraph.
- [ ] I compared seven designs on the same test set and wrote an ADR draft from the numbers.
- [ ] I can say which specialist would gain from a harness, and why.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Re-read the [Multi-agent](https://docs.langchain.com/oss/python/langchain/multi-agent/index) overview and log in [`docs/research-log.md`](../../docs/research-log.md): whether the list of recommended patterns changed (2026-10-12: subagents, handoffs, skills, router, custom workflow) and whether [`langgraph-supervisor`](https://docs.langchain.com/oss/python/migrate/langgraph-supervisor) is still "no longer actively maintained". This is the module's research checkpoint.

## Summary

- A custom workflow is a graph of your own: deterministic steps where rules must hold, model/agent steps where language is needed.
- Pattern C (refunds): parse → validate → approve (interrupt) → issue → reply: one model call, policy in code.
- Workflows embed agents (agent inside a node) and are embedded (subgraph node) with one checkpointer at the top.
- Decide from a table: pass rate, calls, tokens, policy held. Write the ADR; revisit it when the numbers change.
- Harnesses (Module 11) pay off for open-ended, multi-step specialists, not for fixed policies.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: the **module project**: read [project/README.md](project/README.md) (spec, modes, acceptance criteria).

[Module 09 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)